# Advanced K-means Clustering with Comprehensive Visualizations

This notebook provides enhanced K-means clustering implementation with advanced graph visualizations for methodology analysis.

## Features:
- Convergence analysis plots
- Cluster quality metrics (Silhouette scores)
- 2D/3D scatter plots in RGB/HSV color space
- Elbow method for optimal k selection
- Interactive Plotly visualizations
- Performance analysis and comparisons

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import cv2
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
from sklearn.datasets import make_blobs
import pandas as pd
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("All libraries imported successfully!")

## Enhanced K-means Implementation with Tracking

In [ ]:
class EnhancedKMeans:
    """Enhanced K-means with comprehensive tracking and visualization capabilities."""
    
    def __init__(self, n_clusters=3, max_iters=100, random_state=42):
        self.n_clusters = n_clusters
        self.max_iters = max_iters
        self.random_state = random_state
        
        # Tracking variables
        self.inertia_history = []
        self.centroid_history = []
        self.centroid_movement = []
        self.iteration_times = []
        
    def fit(self, X):
        """Fit K-means with detailed tracking."""
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        
        # Initialize centroids randomly
        centroids = X[np.random.choice(n_samples, self.n_clusters, replace=False)]
        self.centroid_history.append(centroids.copy())
        
        for iteration in range(self.max_iters):
            start_time = time.time()
            
            # Assign points to closest centroid
            distances = np.sqrt(((X - centroids[:, np.newaxis])**2).sum(axis=2))
            labels = np.argmin(distances, axis=0)
            
            # Calculate inertia
            inertia = np.sum([np.sum((X[labels == i] - centroids[i])**2) for i in range(self.n_clusters)])
            self.inertia_history.append(inertia)
            
            # Update centroids
            new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(self.n_clusters)])
            
            # Track centroid movement
            if len(self.centroid_history) > 0:
                movement = np.linalg.norm(new_centroids - centroids, axis=1).mean()
                self.centroid_movement.append(movement)
            
            self.centroid_history.append(new_centroids.copy())
            
            # Check convergence
            if np.allclose(centroids, new_centroids, rtol=1e-6):
                break
                
            centroids = new_centroids
            
            # Track iteration time
            self.iteration_times.append(time.time() - start_time)
        
        self.centroids_ = centroids
        self.labels_ = labels
        self.n_iter_ = iteration + 1
        
        return self
    
    def predict(self, X):
        """Predict cluster labels for new data."""
        distances = np.sqrt(((X - self.centroids_[:, np.newaxis])**2).sum(axis=2))
        return np.argmin(distances, axis=0)

print("Enhanced K-means class defined successfully!")

## Convergence Analysis Visualizations

In [ ]:
def plot_convergence_analysis(kmeans_model, title="K-means Convergence Analysis"):
    """Plot comprehensive convergence analysis."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # 1. Inertia over iterations
    axes[0, 0].plot(range(len(kmeans_model.inertia_history)), kmeans_model.inertia_history, 
                    marker='o', linewidth=2, markersize=4)
    axes[0, 0].set_title('Inertia Reduction Over Iterations')
    axes[0, 0].set_xlabel('Iteration')
    axes[0, 0].set_ylabel('Inertia')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Centroid movement over iterations
    if kmeans_model.centroid_movement:
        axes[0, 1].plot(range(len(kmeans_model.centroid_movement)), kmeans_model.centroid_movement, 
                        marker='s', linewidth=2, markersize=4, color='orange')
        axes[0, 1].set_title('Average Centroid Movement')
        axes[0, 1].set_xlabel('Iteration')
        axes[0, 1].set_ylabel('Movement Distance')
        axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Iteration times
    if kmeans_model.iteration_times:
        axes[1, 0].bar(range(len(kmeans_model.iteration_times)), kmeans_model.iteration_times)
        axes[1, 0].set_title('Processing Time per Iteration')
        axes[1, 0].set_xlabel('Iteration')
        axes[1, 0].set_ylabel('Time (seconds)')
        axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Convergence rate (relative inertia change)
    if len(kmeans_model.inertia_history) > 1:
        inertia_changes = []
        for i in range(1, len(kmeans_model.inertia_history)):
            change = abs(kmeans_model.inertia_history[i] - kmeans_model.inertia_history[i-1]) / kmeans_model.inertia_history[i-1]
            inertia_changes.append(change)
        
        axes[1, 1].semilogy(range(len(inertia_changes)), inertia_changes, 
                           marker='^', linewidth=2, markersize=4, color='red')
        axes[1, 1].set_title('Convergence Rate (Log Scale)')
        axes[1, 1].set_xlabel('Iteration')
        axes[1, 1].set_ylabel('Relative Inertia Change')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("Convergence analysis function defined!")

## Cluster Quality Metrics and Visualizations

In [ ]:
def plot_cluster_quality_metrics(X, labels, centroids, title="Cluster Quality Analysis"):
    """Plot comprehensive cluster quality metrics."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # 1. Silhouette analysis
    silhouette_avg = silhouette_score(X, labels)
    sample_silhouette_values = silhouette_samples(X, labels)
    
    y_lower = 10
    for i in range(len(np.unique(labels))):
        cluster_silhouette_values = sample_silhouette_values[labels == i]
        cluster_silhouette_values.sort()
        
        size_cluster_i = cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = plt.cm.nipy_spectral(float(i) / len(np.unique(labels)))
        axes[0, 0].fill_betweenx(np.arange(y_lower, y_upper),
                                 0, cluster_silhouette_values,
                                 facecolor=color, edgecolor=color, alpha=0.7)
        
        axes[0, 0].text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10
    
    axes[0, 0].axvline(x=silhouette_avg, color="red", linestyle="--", 
                       label=f'Average Score: {silhouette_avg:.3f}')
    axes[0, 0].set_title('Silhouette Analysis')
    axes[0, 0].set_xlabel('Silhouette Coefficient Values')
    axes[0, 0].set_ylabel('Cluster Label')
    axes[0, 0].legend()
    
    # 2. Cluster size distribution
    unique_labels, counts = np.unique(labels, return_counts=True)
    axes[0, 1].bar(unique_labels, counts, color=plt.cm.Set3(np.linspace(0, 1, len(unique_labels))))
    axes[0, 1].set_title('Cluster Size Distribution')
    axes[0, 1].set_xlabel('Cluster ID')
    axes[0, 1].set_ylabel('Number of Points')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Within-cluster sum of squares (WCSS) per cluster
    wcss_per_cluster = []
    for i in range(len(np.unique(labels))):
        cluster_points = X[labels == i]
        if len(cluster_points) > 0:
            wcss = np.sum((cluster_points - centroids[i])**2)
            wcss_per_cluster.append(wcss)
        else:
            wcss_per_cluster.append(0)
    
    axes[1, 0].bar(range(len(wcss_per_cluster)), wcss_per_cluster, 
                   color=plt.cm.viridis(np.linspace(0, 1, len(wcss_per_cluster))))
    axes[1, 0].set_title('Within-Cluster Sum of Squares')
    axes[1, 0].set_xlabel('Cluster ID')
    axes[1, 0].set_ylabel('WCSS')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Cluster compactness (average distance to centroid)
    avg_distances = []
    for i in range(len(np.unique(labels))):
        cluster_points = X[labels == i]
        if len(cluster_points) > 0:
            distances = np.linalg.norm(cluster_points - centroids[i], axis=1)
            avg_distances.append(np.mean(distances))
        else:
            avg_distances.append(0)
    
    axes[1, 1].bar(range(len(avg_distances)), avg_distances, 
                   color=plt.cm.plasma(np.linspace(0, 1, len(avg_distances))))
    axes[1, 1].set_title('Average Distance to Centroid')
    axes[1, 1].set_xlabel('Cluster ID')
    axes[1, 1].set_ylabel('Average Distance')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return silhouette_avg

print("Cluster quality metrics function defined!")

## Elbow Method for Optimal K Selection

In [ ]:
def elbow_method_analysis(X, k_range=range(1, 11), title="Elbow Method Analysis"):
    """Comprehensive elbow method analysis with multiple metrics."""
    inertias = []
    silhouette_scores = []
    
    print("Computing metrics for different k values...")
    for k in tqdm(k_range):
        if k == 1:
            # For k=1, inertia is total sum of squares
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            kmeans.fit(X)
            inertias.append(kmeans.inertia_)
            silhouette_scores.append(0)  # Silhouette score undefined for k=1
        else:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            kmeans.fit(X)
            inertias.append(kmeans.inertia_)
            silhouette_scores.append(silhouette_score(X, kmeans.labels_))
    
    # Create comprehensive elbow analysis
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(title, fontsize=16, fontweight='bold')
    
    # 1. Classic elbow curve
    axes[0, 0].plot(k_range, inertias, marker='o', linewidth=2, markersize=6)
    axes[0, 0].set_title('Elbow Curve (Inertia vs K)')
    axes[0, 0].set_xlabel('Number of Clusters (k)')
    axes[0, 0].set_ylabel('Inertia (WCSS)')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Silhouette scores
    k_range_sil = [k for k in k_range if k > 1]
    sil_scores_filtered = [s for s in silhouette_scores if s > 0]
    axes[0, 1].plot(k_range_sil, sil_scores_filtered, marker='s', linewidth=2, markersize=6, color='orange')
    axes[0, 1].set_title('Silhouette Score vs K')
    axes[0, 1].set_xlabel('Number of Clusters (k)')
    axes[0, 1].set_ylabel('Silhouette Score')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Rate of change in inertia
    inertia_changes = []
    for i in range(1, len(inertias)):
        change = inertias[i-1] - inertias[i]
        inertia_changes.append(change)
    
    axes[1, 0].plot(list(k_range)[1:], inertia_changes, marker='^', linewidth=2, markersize=6, color='red')
    axes[1, 0].set_title('Rate of Inertia Reduction')
    axes[1, 0].set_xlabel('Number of Clusters (k)')
    axes[1, 0].set_ylabel('Inertia Reduction')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Combined metric (normalized)
    # Normalize both metrics to 0-1 scale
    norm_inertias = [(max(inertias) - i) / (max(inertias) - min(inertias)) for i in inertias]
    norm_silhouettes = [0] + [(s - min(sil_scores_filtered)) / (max(sil_scores_filtered) - min(sil_scores_filtered)) 
                              for s in sil_scores_filtered]
    
    combined_score = [(norm_inertias[i] + norm_silhouettes[i]) / 2 for i in range(len(norm_inertias))]
    
    axes[1, 1].plot(k_range, combined_score, marker='d', linewidth=2, markersize=6, color='purple')
    axes[1, 1].set_title('Combined Score (Normalized)')
    axes[1, 1].set_xlabel('Number of Clusters (k)')
    axes[1, 1].set_ylabel('Combined Score')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Find optimal k
    optimal_k = k_range[np.argmax(combined_score)]
    print(f"\nRecommended optimal k: {optimal_k}")
    print(f"Inertia at optimal k: {inertias[optimal_k-min(k_range)]:.2f}")
    if optimal_k > 1:
        print(f"Silhouette score at optimal k: {silhouette_scores[optimal_k-min(k_range)]:.3f}")
    
    return optimal_k, inertias, silhouette_scores

print("Elbow method analysis function defined!")

## Interactive 3D Visualizations with Plotly

In [ ]:
def create_interactive_3d_clusters(X, labels, centroids, feature_names=None, title="Interactive 3D Cluster Visualization"):
    """Create interactive 3D scatter plot with Plotly."""
    if X.shape[1] < 3:
        print("Data has less than 3 dimensions. Using PCA for 3D visualization...")
        pca = PCA(n_components=3)
        X_3d = pca.fit_transform(X)
        centroids_3d = pca.transform(centroids)
        feature_names = ['PC1', 'PC2', 'PC3']
    else:
        X_3d = X[:, :3]
        centroids_3d = centroids[:, :3]
        if feature_names is None:
            feature_names = ['Feature 1', 'Feature 2', 'Feature 3']
    
    # Create DataFrame for easier plotting
    df = pd.DataFrame({
        feature_names[0]: X_3d[:, 0],
        feature_names[1]: X_3d[:, 1],
        feature_names[2]: X_3d[:, 2],
        'Cluster': labels.astype(str),
        'Point_ID': range(len(X_3d))
    })
    
    # Create 3D scatter plot
    fig = px.scatter_3d(df, 
                        x=feature_names[0], 
                        y=feature_names[1], 
                        z=feature_names[2],
                        color='Cluster',
                        hover_data=['Point_ID'],
                        title=title,
                        color_discrete_sequence=px.colors.qualitative.Set1)
    
    # Add centroids
    for i, centroid in enumerate(centroids_3d):
        fig.add_trace(go.Scatter3d(
            x=[centroid[0]],
            y=[centroid[1]],
            z=[centroid[2]],
            mode='markers',
            marker=dict(
                size=15,
                symbol='x',
                color='black',
                line=dict(width=2)
            ),
            name=f'Centroid {i}',
            hovertemplate=f'Centroid {i}<br>' +
                         f'{feature_names[0]}: %{{x:.2f}}<br>' +
                         f'{feature_names[1]}: %{{y:.2f}}<br>' +
                         f'{feature_names[2]}: %{{z:.2f}}<extra></extra>'
        ))
    
    # Update layout
    fig.update_layout(
        scene=dict(
            xaxis_title=feature_names[0],
            yaxis_title=feature_names[1],
            zaxis_title=feature_names[2]
        ),
        width=800,
        height=600
    )
    
    fig.show()
    return fig

def create_rgb_hsv_visualization(image_data, labels, centroids):
    """Create RGB and HSV color space visualizations."""
    # Convert RGB to HSV
    hsv_data = cv2.cvtColor(image_data.reshape(-1, 1, 3).astype(np.uint8), cv2.COLOR_RGB2HSV)
    hsv_data = hsv_data.reshape(-1, 3).astype(np.float32)
    
    # Create subplots
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scatter3d'}, {'type': 'scatter3d'}]],
        subplot_titles=['RGB Color Space', 'HSV Color Space']
    )
    
    # Subsample for better performance
    n_samples = min(5000, len(image_data))
    indices = np.random.choice(len(image_data), n_samples, replace=False)
    
    rgb_sample = image_data[indices]
    hsv_sample = hsv_data[indices]
    labels_sample = labels[indices]
    
    # RGB scatter plot
    for cluster_id in np.unique(labels_sample):
        mask = labels_sample == cluster_id
        rgb_cluster = rgb_sample[mask]
        
        fig.add_trace(
            go.Scatter3d(
                x=rgb_cluster[:, 0],
                y=rgb_cluster[:, 1],
                z=rgb_cluster[:, 2],
                mode='markers',
                marker=dict(
                    size=3,
                    color=rgb_cluster / 255.0,
                    opacity=0.6
                ),
                name=f'Cluster {cluster_id}',
                showlegend=True
            ),
            row=1, col=1
        )
    
    # HSV scatter plot
    for cluster_id in np.unique(labels_sample):
        mask = labels_sample == cluster_id
        hsv_cluster = hsv_sample[mask]
        
        fig.add_trace(
            go.Scatter3d(
                x=hsv_cluster[:, 0],
                y=hsv_cluster[:, 1],
                z=hsv_cluster[:, 2],
                mode='markers',
                marker=dict(
                    size=3,
                    opacity=0.6
                ),
                name=f'Cluster {cluster_id} (HSV)',
                showlegend=False
            ),
            row=1, col=2
        )
    
    # Update layout
    fig.update_layout(
        scene=dict(
            xaxis_title='Red',
            yaxis_title='Green',
            zaxis_title='Blue'
        ),
        scene2=dict(
            xaxis_title='Hue',
            yaxis_title='Saturation',
            zaxis_title='Value'
        ),
        width=1200,
        height=600,
        title='Image Clustering in RGB and HSV Color Spaces'
    )
    
    fig.show()
    return fig

print("Interactive visualization functions defined!")

## Demonstration with Sample Data

In [ ]:
# Create sample data for demonstration
print("Creating sample data for demonstration...")

# Generate synthetic 2D data
X_2d, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, 
                          center_box=(-10.0, 10.0), random_state=42)

# Generate sample 3D data
X_3d, _ = make_blobs(n_samples=500, centers=3, n_features=3, 
                     cluster_std=1.2, random_state=42)

print(f"2D data shape: {X_2d.shape}")
print(f"3D data shape: {X_3d.shape}")
print("Sample data created successfully!")

In [ ]:
# Demonstrate enhanced K-means on 2D data
print("Running enhanced K-means on 2D data...")

enhanced_kmeans = EnhancedKMeans(n_clusters=4, max_iters=50)
enhanced_kmeans.fit(X_2d)

print(f"Converged in {enhanced_kmeans.n_iter_} iterations")
print(f"Final inertia: {enhanced_kmeans.inertia_history[-1]:.2f}")

# Plot convergence analysis
plot_convergence_analysis(enhanced_kmeans, "Enhanced K-means Convergence (2D Data)")

# Plot cluster quality metrics
silhouette_avg = plot_cluster_quality_metrics(X_2d, enhanced_kmeans.labels_, 
                                              enhanced_kmeans.centroids_, 
                                              "Cluster Quality Analysis (2D Data)")

print(f"Average silhouette score: {silhouette_avg:.3f}")

In [ ]:
# Demonstrate elbow method
print("Running elbow method analysis...")
optimal_k, inertias, silhouette_scores = elbow_method_analysis(X_2d, range(1, 10), 
                                                               "Elbow Method Analysis (2D Data)")

In [ ]:
# Demonstrate 3D interactive visualization
print("Creating interactive 3D visualization...")

# Run K-means on 3D data
kmeans_3d = KMeans(n_clusters=3, random_state=42)
labels_3d = kmeans_3d.fit_predict(X_3d)

# Create interactive plot
fig_3d = create_interactive_3d_clusters(X_3d, labels_3d, kmeans_3d.cluster_centers_,
                                        ['X', 'Y', 'Z'], "Interactive 3D K-means Clustering")

print("Interactive 3D visualization created!")

## Image Clustering Demonstration

This section demonstrates K-means clustering on the provided test image with advanced visualizations.

In [ ]:
# Load and process the test image
try:
    # Try to load the image from the repository
    image_path = '/home/runner/work/pfe/pfe/imagetest.jpg'
    image = cv2.imread(image_path)
    if image is not None:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        print(f"Loaded image with shape: {image.shape}")
        
        # Display original image
        plt.figure(figsize=(8, 6))
        plt.imshow(image)
        plt.title('Original Image')
        plt.axis('off')
        plt.show()
        
        # Prepare pixel data
        pixel_values = image.reshape((-1, 3)).astype(np.float32)
        print(f"Pixel data shape: {pixel_values.shape}")
        
    else:
        print("Could not load image. Creating synthetic image data...")
        # Create synthetic image data
        image = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)
        pixel_values = image.reshape((-1, 3)).astype(np.float32)
        
except Exception as e:
    print(f"Error loading image: {e}")
    print("Creating synthetic image data...")
    # Create synthetic image data
    image = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)
    pixel_values = image.reshape((-1, 3)).astype(np.float32)

In [ ]:
# Perform K-means clustering on image
print("Performing K-means clustering on image...")

# Find optimal k for the image
optimal_k_img, _, _ = elbow_method_analysis(pixel_values, range(2, 8), 
                                           "Elbow Method for Image Clustering")

# Run enhanced K-means with optimal k
enhanced_kmeans_img = EnhancedKMeans(n_clusters=optimal_k_img, max_iters=30)
enhanced_kmeans_img.fit(pixel_values)

print(f"Image clustering completed with k={optimal_k_img}")
print(f"Converged in {enhanced_kmeans_img.n_iter_} iterations")

# Plot convergence analysis for image
plot_convergence_analysis(enhanced_kmeans_img, "Image Clustering Convergence Analysis")

# Plot cluster quality metrics for image
silhouette_avg_img = plot_cluster_quality_metrics(pixel_values, enhanced_kmeans_img.labels_, 
                                                  enhanced_kmeans_img.centroids_, 
                                                  "Image Clustering Quality Analysis")

print(f"Image clustering silhouette score: {silhouette_avg_img:.3f}")

In [ ]:
# Visualize clustering results
print("Creating clustered image visualization...")

# Create segmented image
centers = enhanced_kmeans_img.centroids_.astype(np.uint8)
segmented_image = centers[enhanced_kmeans_img.labels_]
segmented_image = segmented_image.reshape(image.shape)

# Display original and segmented images
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(image)
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(segmented_image)
axes[1].set_title(f'K-means Segmentation (k={optimal_k_img})')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# Create RGB/HSV color space visualization
if len(pixel_values) > 1000:  # Only for reasonable sized images
    print("Creating RGB and HSV color space visualizations...")
    fig_rgb_hsv = create_rgb_hsv_visualization(pixel_values, enhanced_kmeans_img.labels_, 
                                              enhanced_kmeans_img.centroids_)

print("Image clustering demonstration completed!")

## Summary and Conclusions

This notebook demonstrates advanced K-means clustering with comprehensive visualizations including:

1. **Convergence Analysis**: Tracks inertia reduction, centroid movement, and processing times
2. **Cluster Quality Metrics**: Silhouette analysis, cluster size distribution, and compactness measures
3. **Optimal K Selection**: Multi-metric elbow method analysis
4. **Interactive 3D Visualizations**: Plotly-based interactive scatter plots
5. **Color Space Analysis**: RGB and HSV visualizations for image clustering

These tools provide comprehensive insight into clustering behavior and quality, making them invaluable for methodology research and practical applications.

### Key Features:
- **Google Colab Compatible**: All visualizations work in Google Colab environment
- **Performance Optimized**: Efficient implementations for large datasets
- **Extensible**: Easy to modify for different applications
- **Well Documented**: Clear explanations and usage examples